In [2]:
# ==========================================
# 1. IMPORTAÇÃO E CARREGAMENTO CORRIGIDO
# ==========================================
import pandas as pd
import numpy as np
from scipy.stats import shapiro, levene, f_oneway, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd

marcas = ['aurora', 'bravura', 'cerrado', 'delta', 'estacao', 'flora', 'granito', 'horizonte']
dfs = []

for i, marca in enumerate(marcas, start=1):
    # Ajustado para procurar direto na raiz onde os arquivos estão
    nome_arq = f"{str(i).zfill(2)}_{marca}.csv"
    temp_df = pd.read_csv(nome_arq)

    if 'marca' not in temp_df.columns:
        temp_df['marca'] = marca.capitalize()
    dfs.append(temp_df)

# Unificando as bases (5.120 registros no total)
df_total = pd.concat(dfs, ignore_index=True)
print(f"✅ Dataset unificado com sucesso! Total de linhas: {df_total.shape[0]}")
display(df_total.head())

✅ Dataset unificado com sucesso! Total de linhas: 5120


,capsula_id,marca,material_embalagem,massa_cafe_g,massa_embalagem_g,co2e_producao_g,co2e_logistica_g,co2e_fim_vida_g,co2e_total_g,agua_total_l,...,distancia_logistica_km,auditoria_fornecedor_realizada,nao_conformidade_trabalhista,trabalho_infantil_constatado,remuneracao_relativa_referencia,desmatamento_atribuido_m2,indice_efluentes_toxicos,coleta_pos_consumo,destinacao_final,reciclagem_efetiva
0,AUR-00001,Aurora,aluminio,3.84,6.98,148.45,46.98,45.21,240.64,13.35,...,206.6,1,0.0,0.0,0.671,0.650,0.611,0,aterro_ou_rejeito,0
1,AUR-00002,Aurora,aluminio,5.80,7.11,107.80,49.64,50.64,208.08,19.02,...,271.9,1,0.0,0.0,0.638,1.544,0.703,1,reciclagem,1
2,AUR-00003,Aurora,aluminio,5.95,6.58,112.35,27.43,44.05,183.83,11.67,...,344.7,1,0.0,0.0,0.540,0.281,0.550,0,aterro_ou_rejeito,0
3,AUR-00004,Aurora,aluminio,5.44,7.37,110.68,38.18,38.35,187.21,9.81,...,296.3,0,NaN,NaN,0.750,0.266,0.910,1,aterro_ou_rejeito,0
4,AUR-00005,Aurora,aluminio,6.59,6.10,136.46,30.20,44.78,211.44,10.57,...,241.0,0,NaN,NaN,0.561,0.131,0.972,1,reciclagem,1


In [3]:
# ==========================================
# 2. ANÁLISE EXPLORATÓRIA RÁPIDA (Ex: CO2e Total)
# ==========================================
# Vamos analisar a variável 'co2e_total_g' (já agrega produção, logística e fim de vida, conforme o dicionário de dados)
metrica = 'co2e_total_g'

print("Estatísticas descritivas por marca (Média e Desvio Padrão):")
resumo = df_total.groupby('marca')[metrica].agg(['count', 'mean', 'std', 'median']).reset_index()
display(resumo)

Estatísticas descritivas por marca (Média e Desvio Padrão):


,marca,count,mean,std,median
0,Aurora,640,179.241125,24.422796,178.640
1,Bravura,640,145.055516,24.221675,143.725
2,Cerrado,640,155.189797,23.611817,155.670
3,Delta,640,215.154297,24.864443,216.285
4,Estacao,640,164.633141,23.380555,163.090
5,Flora,640,168.693172,24.517326,167.545
6,Granito,640,194.795469,25.267061,194.155
7,Horizonte,640,173.572250,25.104985,175.485


In [4]:
# ==========================================
# 3. VERIFICAÇÃO DE PRESSUPOSTOS (Shapiro e Levene)
# ==========================================
alpha = 0.05
grupos = [group[metrica].values for name, group in df_total.groupby('marca')]

# Teste de Normalidade (Shapiro-Wilk) para cada marca
normais = True
print("--- Teste de Shapiro-Wilk (Normalidade) ---")
for i, marca_nome in enumerate(df_total['marca'].unique()):
    stat_shap, p_shap = shapiro(grupos[i][:500]) # Amostra limitada se necessário
    print(f"Marca {marca_nome}: p-valor = {p_shap:.5f}")
    if p_shap < alpha:
        normais = False

# Teste de Homocedasticidade (Levene)
stat_lev, p_lev = levene(*grupos)
print(f"\n--- Teste de Levene (Variâncias iguais) ---\np-valor = {p_lev:.5f}")

if normais and p_lev >= alpha:
    print("\n👉 Condições atendidas: Os dados são normais e possuem variâncias homogêneas. Usaremos ANOVA.")
elif normais and p_lev < alpha:
    print("\n👉 Condições: Dados normais, mas variâncias heterogêneas. Recomendado usar Welch ANOVA.")
else:
    print("\n👉 Condições: Dados NÃO seguem a distribuição normal. Usaremos Kruskal-Wallis (Não paramétrico).")

--- Teste de Shapiro-Wilk (Normalidade) ---
Marca Aurora: p-valor = 0.65191
Marca Bravura: p-valor = 0.55711
Marca Cerrado: p-valor = 0.27935
Marca Delta: p-valor = 0.90284
Marca Estacao: p-valor = 0.13790
Marca Flora: p-valor = 0.34127
Marca Granito: p-valor = 0.57321
Marca Horizonte: p-valor = 0.18712

--- Teste de Levene (Variâncias iguais) ---
p-valor = 0.37559

👉 Condições atendidas: Os dados são normais e possuem variâncias homogêneas. Usaremos ANOVA.


In [5]:
# ==========================================
# 4. APLICAÇÃO DO TESTE GLOBAL (ANOVA vs Kruskal)
# ==========================================
# Definindo H0 e H1:
# H0: As emissões médias (ou distribuições) de CO2e total são iguais entre todas as marcas.
# H1: Pelo menos uma marca apresenta diferença significativa nas emissões.

if normais and p_lev >= alpha:
    stat_global, p_global = f_oneway(*grupos)
    teste_usado = "ANOVA"
elif normais and p_lev < alpha:
    import pingouin as pg
    welch_res = pg.welch_anova(dv=metrica, between='marca', data=df_total)
    stat_global, p_global = welch_res['F'].values[0], welch_res['p-val'].values[0]
    teste_usado = "Welch ANOVA"
else:
    stat_global, p_global = kruskal(*grupos)
    teste_usado = "Kruskal-Wallis"

print(f"Teste Global ({teste_usado}):")
print(f"Estatística = {stat_global:.4f} | p-valor = {p_global:.4e}")

if p_global < alpha:
    print("Resultado: Rejeitamos H0 (Há diferença estatisticamente significativa entre as marcas!).")
else:
    print("Resultado: Não há evidências suficientes para rejeitar H0.")

Teste Global (ANOVA):
Estatística = 529.7965 | p-valor = 0.0000e+00
Resultado: Rejeitamos H0 (Há diferença estatisticamente significativa entre as marcas!).


In [6]:
# ==========================================
# 5. TESTE POST-HOC (Tukey HSD ou Dunn)
# ==========================================
# Se o teste global deu significativo, precisamos descobrir quais marcas diferem entre si.

if p_global < alpha:
    if "Kruskal" not in teste_usado:
        # Post-hoc de Tukey para ANOVA
        tukey = pairwise_tukeyhsd(endog=df_total[metrica], groups=df_total['marca'], alpha=alpha)
        print("\n--- Resultados do Post-hoc de Tukey HSD ---")
        print(tukey)
    else:
        # Post-hoc de Dunn para Kruskal-Wallis
        !pip install -q scikit-posthocs
        import scikit_posthocs as sp
        dunn = sp.posthoc_dunn(df_total, val_col=metrica, group_col='marca', p_adjust='bonferroni')
        print("\n--- Resultados do Post-hoc de Dunn (com ajuste de Bonferroni) ---")
        display(dunn)
else:
    print("Como o teste global não foi significativo, o post-hoc não é necessário.")


--- Resultados do Post-hoc de Tukey HSD ---
   Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1   group2  meandiff p-adj   lower    upper   reject
----------------------------------------------------------
 Aurora   Bravura -34.1856    0.0 -38.3269 -30.0444   True
 Aurora   Cerrado -24.0513    0.0 -28.1926 -19.9101   True
 Aurora     Delta  35.9132    0.0  31.7719  40.0544   True
 Aurora   Estacao  -14.608    0.0 -18.7492 -10.4667   True
 Aurora     Flora  -10.548    0.0 -14.6892  -6.4067   True
 Aurora   Granito  15.5543    0.0  11.4131  19.6956   True
 Aurora Horizonte  -5.6689 0.0009  -9.8101  -1.5276   True
Bravura   Cerrado  10.1343    0.0    5.993  14.2755   True
Bravura     Delta  70.0988    0.0  65.9575    74.24   True
Bravura   Estacao  19.5776    0.0  15.4364  23.7189   True
Bravura     Flora  23.6377    0.0  19.4964  27.7789   True
Bravura   Granito    49.74    0.0  45.5987  53.8812   True
Bravura Horizonte  28.5167    0.0  24.3755   32.658   True
Cerrado    

In [7]:
# ==========================================
# PANORAMA MULTIDIMENSIONAL POR MARCA
# ==========================================

# 1. Agrupando médias das principais variáveis ambientais e sociais
metricas_alvo = [
    'co2e_total_g',
    'agua_total_l',
    'desmatamento_atribuido_m2',
    'remuneracao_relativa_referencia',
    'reciclagem_efetiva'
]

resumo_multidimensional = df_total.groupby('marca')[metricas_alvo].mean().reset_index()

print("📊 MÉDIAS GERAIS POR MARCA (Ambiental e Social):")
display(resumo_multidimensional.round(2))

# 2. Verificando dados de auditoria e irregularidades (lembrando que vazio != 0, conforme o dicionário)
print("\n🔍 ANÁLISE DE AUDITORIAS E IRREGULARIDADES (Contagem de casos '1'):")
auditoria_resumo = df_total.groupby('marca').agg(
    total_capsulas=('capsula_id', 'count'),
    auditorias_realizadas=('auditoria_fornecedor_realizada', 'sum'),
    irregularidade_trabalhista=('nao_conformidade_trabalhista', 'sum'),
    trabalho_infantil=('trabalho_infantil_constatado', 'sum')
).reset_index()

display(auditoria_resumo)

📊 MÉDIAS GERAIS POR MARCA (Ambiental e Social):


,marca,co2e_total_g,agua_total_l,desmatamento_atribuido_m2,remuneracao_relativa_referencia,reciclagem_efetiva
0,Aurora,179.24,12.64,1.13,0.68,0.29
1,Bravura,145.06,15.82,1.55,0.66,0.43
2,Cerrado,155.19,20.28,1.43,0.69,0.29
3,Delta,215.15,11.15,1.00,0.69,0.21
4,Estacao,164.63,17.17,1.70,0.65,0.35
5,Flora,168.69,19.59,1.32,0.67,0.30
6,Granito,194.80,12.25,1.53,0.66,0.26
7,Horizonte,173.57,15.65,1.68,0.67,0.31



🔍 ANÁLISE DE AUDITORIAS E IRREGULARIDADES (Contagem de casos '1'):


,marca,total_capsulas,auditorias_realizadas,irregularidade_trabalhista,trabalho_infantil
0,Aurora,640,498,116.0,43.0
1,Bravura,640,468,91.0,56.0
2,Cerrado,640,482,126.0,73.0
3,Delta,640,480,151.0,30.0
4,Estacao,640,171,44.0,36.0
5,Flora,640,481,100.0,53.0
6,Granito,640,477,106.0,60.0
7,Horizonte,640,481,82.0,67.0


In [8]:
# ==========================================
# TESTES ESTATÍSTICOS ADICIONAIS (Água e Desmatamento)
# ==========================================
from scipy.stats import f_oneway, kruskal, shapiro, levene, pearsonr
from statsmodels.stats.multicomp import pairwise_tukeyhsd

alpha = 0.05

# 1. TESTE PARA PEGADA HÍDRICA (agua_total_l)
print("💧 --- ANÁLISE DA PEGADA HÍDRICA (agua_total_l) ---")
grupos_agua = [group['agua_total_l'].dropna().values for name, group in df_total.groupby('marca')]
stat_a, p_agua = f_oneway(*grupos_agua)
print(f"ANOVA para Água: Estatística = {stat_a:.4f} | p-valor = {p_agua:.4e}")

if p_agua < alpha:
    print("👉 Há diferença significativa na pegada hídrica entre as marcas.")
    # Post-hoc de Tukey para água
    tukey_agua = pairwise_tukeyhsd(endog=df_total['agua_total_l'], groups=df_total['marca'], alpha=alpha)
    print(tukey_agua)

print("\n" + "="*50 + "\n")

# 2. TESTE PARA DESMATAMENTO ATRIBUÍDO (desmatamento_atribuido_m2)
print("🌳 --- ANÁLISE DE DESMATAMENTO ATRIBUÍDO (desmatamento_atribuido_m2) ---")
grupos_desm = [group['desmatamento_atribuido_m2'].dropna().values for name, group in df_total.groupby('marca')]
stat_d, p_desm = f_oneway(*grupos_desm)
print(f"ANOVA para Desmatamento: Estatística = {stat_d:.4f} | p-valor = {p_desm:.4e}")

if p_desm < alpha:
    print("👉 Há diferença significativa no desmatamento atribuído entre as marcas.")

print("\n" + "="*50 + "\n")

# 3. CORRELAÇÃO ENTRE CARBONO E ÁGUA
corr, p_corr = pearsonr(df_total['co2e_total_g'], df_total['agua_total_l'])
print(f"🔗 Correlação de Pearson entre Carbono e Água: r = {corr:.4f} (p-valor = {p_corr:.4e})")

💧 --- ANÁLISE DA PEGADA HÍDRICA (agua_total_l) ---
ANOVA para Água: Estatística = 163.6070 | p-valor = 5.7331e-219
👉 Há diferença significativa na pegada hídrica entre as marcas.
   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1   group2  meandiff p-adj   lower    upper  reject
---------------------------------------------------------
 Aurora   Bravura   3.1748    0.0   2.0407  4.3088   True
 Aurora   Cerrado   7.6352    0.0   6.5011  8.7693   True
 Aurora     Delta  -1.4963 0.0016  -2.6304 -0.3622   True
 Aurora   Estacao   4.5255    0.0   3.3914  5.6595   True
 Aurora     Flora   6.9468    0.0   5.8127  8.0808   True
 Aurora   Granito  -0.3932 0.9664  -1.5272  0.7409  False
 Aurora Horizonte   3.0068    0.0   1.8727  4.1409   True
Bravura   Cerrado   4.4604    0.0   3.3264  5.5945   True
Bravura     Delta  -4.6711    0.0  -5.8052  -3.537   True
Bravura   Estacao   1.3507 0.0074   0.2166  2.4848   True
Bravura     Flora    3.772    0.0   2.6379  4.9061   True
Bravura  

In [9]:
# =========================================================
# BATERIA AVANÇADA DE TESTES: EFLUENTES, SOCIAL E SCORE GLOBAL
# =========================================================
import pandas as pd
import numpy as np
from scipy.stats import kruskal, f_oneway, chi2_contingency
from sklearn.preprocessing import MinMaxScaler

# 1. TESTE DE REMUNERAÇÃO RELATIVA (remuneracao_relativa_referencia)
print("💰 --- TESTE DE REMUNERAÇÃO RELATIVA ---")
grupos_rem = [group['remuneracao_relativa_referencia'].dropna().values for name, group in df_total.groupby('marca')]
stat_rem, p_rem = f_oneway(*grupos_rem)
print(f"ANOVA Remuneração: F = {stat_rem:.4f} | p-valor = {p_rem:.4e}")

# 2. TESTE DE EFLUENTES TÓXICOS (indice_efluentes_toxicos)
print("\n🧪 --- TESTE DE EFLUENTES TÓXICOS ---")
grupos_efl = [group['indice_efluentes_toxicos'].dropna().values for name, group in df_total.groupby('marca')]
stat_efl, p_efl = kruskal(*grupos_efl)
print(f"Kruskal-Wallis Efluentes: H = {stat_efl:.4f} | p-valor = {p_efl:.4e}")

# 3. CONVERSÃO DE COLETA EM RECICLAGEM EFETIVA (Qui-Quadrado)
print("\n♻️ --- TAXA DE CONVERSÃO: COLETA -> RECICLAGEM EFETIVA ---")
df_coletadas = df_total[df_total['coleta_pos_consumo'] == 1]
conversao = df_coletadas.groupby('marca')['reciclagem_efetiva'].agg(['count', 'sum', 'mean']).reset_index()
conversao.columns = ['marca', 'coletadas', 'recicladas_efetivas', 'taxa_conversao']
display(conversao.round(3))

# 4. PROPORÇÃO DE TRABALHO INFANTIL NAS AUDITORIAS (Proporção de 1s sobre auditados)
print("\n⚠️ --- INCIDÊNCIA DE TRABALHO INFANTIL POR AUDITORIA ---")
aud_df = df_total[df_total['auditoria_fornecedor_realizada'] == 1]
prop_ti = aud_df.groupby('marca')['trabalho_infantil_constatado'].agg(['count', 'sum', 'mean']).reset_index()
prop_ti.columns = ['marca', 'auditadas', 'casos_ti', 'taxa_incidencia_ti']
display(prop_ti.round(3))

# =========================================================
# 5. MODELO DE RANKING MULTICRITÉRIO (Z-SCORE / MIN-MAX)
# =========================================================
print("\n🏆 --- RANKING MULTICRITÉRIO (PONDERADO) ---")
df_rank = df_total.groupby('marca').agg(
    co2e=('co2e_total_g', 'mean'),
    agua=('agua_total_l', 'mean'),
    desmatamento=('desmatamento_atribuido_m2', 'mean'),
    efluentes=('indice_efluentes_toxicos', 'mean'),
    remuneracao=('remuneracao_relativa_referencia', 'mean'),
    reciclagem=('reciclagem_efetiva', 'mean')
).reset_index()

# Cruzando com taxa de trabalho infantil
df_rank = df_rank.merge(prop_ti[['marca', 'taxa_incidencia_ti']], on='marca')

# Normalizando (0 a 1) para comparar variáveis de escalas diferentes
scaler = MinMaxScaler()
cols_norm = ['co2e', 'agua', 'desmatamento', 'efluentes', 'remuneracao', 'reciclagem', 'taxa_incidencia_ti']
df_norm = pd.DataFrame(scaler.fit_transform(df_rank[cols_norm]), columns=cols_norm)
df_norm['marca'] = df_rank['marca']

# Quanto MENOR o carbono, água, desmatamento, efluentes e trabalho infantil, MELHOR.
# Quanto MAIOR a remuneração e reciclagem, MELHOR.
df_norm['score_impacto_negativo'] = (
    df_norm['co2e'] +
    df_norm['agua'] +
    df_norm['desmatamento'] +
    df_norm['efluentes'] +
    df_norm['taxa_incidencia_ti']
)
df_norm['score_impacto_positivo'] = df_norm['remuneracao'] + df_norm['reciclagem']

# Score Final sustentável (Menor pontuação = Menor impacto nocivo geral)
df_norm['score_global'] = df_norm['score_impacto_negativo'] - df_norm['score_impacto_positivo']
ranking_final = df_norm[['marca', 'score_global', 'co2e', 'agua', 'taxa_incidencia_ti', 'reciclagem']].sort_values('score_global')

display(ranking_final.round(3))

💰 --- TESTE DE REMUNERAÇÃO RELATIVA ---
ANOVA Remuneração: F = 5.3035 | p-valor = 4.6741e-06

🧪 --- TESTE DE EFLUENTES TÓXICOS ---
Kruskal-Wallis Efluentes: H = 486.2178 | p-valor = 7.3545e-101

♻️ --- TAXA DE CONVERSÃO: COLETA -> RECICLAGEM EFETIVA ---


,marca,coletadas,recicladas_efetivas,taxa_conversao
0,Aurora,385,188,0.488
1,Bravura,455,273,0.600
2,Cerrado,373,183,0.491
3,Delta,330,132,0.400
4,Estacao,423,224,0.530
5,Flora,386,194,0.503
6,Granito,360,168,0.467
7,Horizonte,406,201,0.495



⚠️ --- INCIDÊNCIA DE TRABALHO INFANTIL POR AUDITORIA ---


,marca,auditadas,casos_ti,taxa_incidencia_ti
0,Aurora,498,43.0,0.086
1,Bravura,468,56.0,0.120
2,Cerrado,482,73.0,0.151
3,Delta,480,30.0,0.062
4,Estacao,171,36.0,0.211
5,Flora,481,53.0,0.110
6,Granito,477,60.0,0.126
7,Horizonte,481,67.0,0.139



🏆 --- RANKING MULTICRITÉRIO (PONDERADO) ---


,marca,score_global,co2e,agua,taxa_incidencia_ti,reciclagem
3,Delta,0.000,1.000,0.000,0.000,0.000
0,Aurora,0.015,0.488,0.164,0.161,0.397
1,Bravura,1.217,0.000,0.512,0.386,1.000
2,Cerrado,1.357,0.145,1.000,0.601,0.362
5,Flora,1.431,0.337,0.925,0.322,0.440
7,Horizonte,1.998,0.407,0.493,0.519,0.489
6,Granito,2.018,0.710,0.121,0.428,0.255
4,Estacao,3.286,0.279,0.659,1.000,0.652


In [10]:
# =========================================================
# TESTES DE HIPÓTESES FINAIS: TRABALHO INFANTIL E EMBALAGENS
# =========================================================
import scipy.stats as stats
import pandas as pd

alpha = 0.05

# ---------------------------------------------------------
# HIPÓTESE 1: Qui-Quadrado de Independência (Marca vs Trabalho Infantil)
# H0: A ocorrência de trabalho infantil é independente da marca (distribuição homogênea).
# H1: A ocorrência de trabalho infantil depende significativamente da marca contratada.
# ---------------------------------------------------------
print("⚠️ --- TESTE QUI-QUADRADO: MARCA VS TRABALHO INFANTIL ---")
tabela_ti = pd.crosstab(
    df_total['marca'],
    df_total['trabalho_infantil_constatado'].dropna()
)
chi2, p_chi2, dof, ex = stats.chi2_contingency(tabela_ti)

print(f"Estatística Chi2 = {chi2:.4f} | p-valor = {p_chi2:.4e} | Graus de liberdade = {dof}")

if p_chi2 < alpha:
    print("👉 Resultado: Rejeitamos H0. A incidência de trabalho infantil independe do acaso e varia criticamente de acordo com a marca!")
else:
    print("👉 Resultado: Não há diferença significativa na proporção de trabalho infantil.")

print("\n" + "="*60 + "\n")

# ---------------------------------------------------------
# HIPÓTESE 2: ANOVA para Emissões de Fim de Vida (co2e_fim_vida_g) por Material
# H0: As emissões médias no fim de vida são iguais entre os materiais de embalagem.
# H1: Pelo menos um material apresenta emissão no fim de vida significativamente diferente.
# ---------------------------------------------------------
print("📦 --- TESTE DE FIM DE VIDA (co2e_fim_vida_g) POR MATERIAL ---")
grupos_mat = [group['co2e_fim_vida_g'].dropna().values for name, group in df_total.groupby('material_embalagem')]
stat_mat, p_mat = stats.f_oneway(*grupos_mat)

print(f"ANOVA Fim de Vida por Material: F = {stat_mat:.4f} | p-valor = {p_mat:.4e}")

for mat, group in df_total.groupby('material_embalagem'):
    print(f"Material: {mat:<12} | Média CO2e Fim de Vida = {group['co2e_fim_vida_g'].mean():.2f}g")

⚠️ --- TESTE QUI-QUADRADO: MARCA VS TRABALHO INFANTIL ---
Estatística Chi2 = 40.8728 | p-valor = 8.5640e-07 | Graus de liberdade = 7
👉 Resultado: Rejeitamos H0. A incidência de trabalho infantil independe do acaso e varia criticamente de acordo com a marca!


📦 --- TESTE DE FIM DE VIDA (co2e_fim_vida_g) POR MATERIAL ---
ANOVA Fim de Vida por Material: F = 0.3207 | p-valor = 8.1044e-01
Material: aluminio     | Média CO2e Fim de Vida = 34.86g
Material: compostavel  | Média CO2e Fim de Vida = 35.01g
Material: fibra_mista  | Média CO2e Fim de Vida = 34.55g
Material: plastico     | Média CO2e Fim de Vida = 34.91g
